In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from skimage.io import imread
from skimage.measure import label, regionprops

from scipy.ndimage import gaussian_filter, gaussian_laplace
from skimage.filters import sobel

import json

### In this notebook, we calculate the mean/std/distributions of sizes of the 3 classes: granule, tail, spike

In [ ]:
data_dir = Path("/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images")
labels_dir = data_dir / "labels"
images_dir = data_dir / "rare_granules"

In [ ]:
labels_dict = {
    1: "granule",
    2: "tail",
    3: "spike"
}

In [ ]:
[p for p in data_dir.iterdir() if p.is_dir()]

[PosixPath('/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images/labels'),
 PosixPath('/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images/rare_granules'),
 PosixPath('/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images/2951'),
 PosixPath('/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images/annotations_rare_granules')]

In [ ]:
names_to_exclude = ["40135_J5_new_x4300_-10um_0002.png",
                    "40135_G1_x4300_-10um_0095.png"]

filenames = []
for file in labels_dir.glob("*.png"):
    if file.name in names_to_exclude:
        continue
    filenames.append(file.name)

filenames

['40135_E1_x4300_-10um_0013.png',
 '40135_J5_new_x4300_-10um_0081.png',
 '40135_G1_x4300_-10um_0002.png',
 '40135_G1_x4300_-10um_0101.png',
 '40135_E4_new_x4300_-10um_0053.png',
 '40135_E4_new_x4300_-10um_0023.png',
 '40135_G1_x4300_-10um_0093.png',
 '40135_E4_new_x4300_-10um_0044.png',
 '40135_E4_new_x4300_-10um_0064.png',
 '40135_E4_new_x4300_-10um_0021.png',
 '40135_G1_x4300_-10um_0033.png',
 '40135-B5_x4300_-10um_0025.png',
 '40135_G1_x4300_-10um_0024.png',
 '40135_E4_new_x4300_-10um_0029.png',
 '40135_E4_new_x4300_-10um_0001.png',
 '40135_G1_x4300_-10um_0091.png',
 '40135_E4_new_x4300_-10um_0097.png',
 '40135_E4_new_x4300_-10um_0027.png',
 '40135_G1_x4300_-10um_0027.png',
 '40135_E4_new_x4300_-10um_0082.png',
 '40135_G1_x4300_-10um_0018.png']

In [ ]:
def get_region_characteristics(labels_dir):

    sizes_dict = {}
    axes_dict = {}

    for f in list(labels_dir.glob("*.png")):
        file = f.name
        labels = imread(f"{labels_dir}/{file}")

        for class_value in np.unique(labels):
            if class_value == 0:
                continue

            binary = labels == class_value
            connected = label(binary)

            for region in regionprops(connected):

                # find the area of the regions
                sizes_dict.setdefault(class_value, []).append(region.area)

                # find the major and minor axes
                axes_dict.setdefault(
                    class_value,
                    {"major": [], "minor": []}
                )
        
                axes_dict[class_value]["major"].append(region.axis_major_length)
                axes_dict[class_value]["minor"].append(region.axis_minor_length)

    return sizes_dict, axes_dict

In [ ]:
sizes, axes = get_region_characteristics(labels_dir)

In [ ]:
stats = {}

for key in sizes:
    class_name = labels_dict[key]

    stats[class_name] = {
        "area": {
            "mean": float(np.mean(sizes[key])),
            "std": float(np.std(sizes[key])),
        },
        "major": {
            "mean": float(np.mean(axes[key]["major"])),
            "std": float(np.std(axes[key]["major"])),
        },
        "minor": {
            "mean": float(np.mean(axes[key]["minor"])),
            "std": float(np.std(axes[key]["minor"])),
        },
    }

stats

{'granule': {'area': {'mean': 3023.387755102041, 'std': 1424.5535042183128},
  'major': {'mean': 61.7378022513564, 'std': 14.393787620831342},
  'minor': {'mean': 59.261608845675234, 'std': 13.969764691541528}},
 'tail': {'area': {'mean': 5164.75, 'std': 1701.2770460745069},
  'major': {'mean': 135.5374821410419, 'std': 32.46597350297029},
  'minor': {'mean': 53.49986584526819, 'std': 11.57409858121953}},
 'spike': {'area': {'mean': 3468.222222222222, 'std': 827.2504360534465},
  'major': {'mean': 70.23458449459747, 'std': 8.674841306071444},
  'minor': {'mean': 62.524790022941524, 'std': 8.480135598165038}}}

In [ ]:
# Function to go through all the filenames, extract their cell bounds in a list, then make crops from the image and labels.
# After that the pixel feature extractor will be run on the crops

def crop_extractor(filenames, images_dir, labels_dir, csv_name):

    df = pd.read_csv(images_dir / csv_name)
    columns = ['x topleft pixels', 'y topleft pixels', 'width pixels', 'height pixels']

    crops_dict = {}

    for file in filenames:

        print(file)
        img_crops = []
        label_crops = []
        img = imread(images_dir / file)
        labels = imread(labels_dir / file)

        df_cells = df[
            df["image name"].str.replace(r"\.ome\.tiff?$", "", regex=True) == file.removesuffix(".png")
        ]

        df_cells = df_cells.drop_duplicates(subset=columns, keep='last')
        df_cells = df_cells[df_cells['class name'] == 'cell']

        for _, row in df_cells.iterrows():
            x_tl = round(row["x topleft pixels"])
            y_tl = round(row["y topleft pixels"])
            w = round(row["width pixels"])
            h = round(row["height pixels"])

            if h > 500 and w > 500:
                img_crops.append(img[y_tl:y_tl + h, x_tl:x_tl + w])
                label_crops.append(labels[y_tl:y_tl + h, x_tl:x_tl + w])

        crops_dict[file] = {'img_crops': img_crops, 'label_crops': label_crops}

        del img, labels


    return crops_dict

In [ ]:
csv_name = "rare_granules_raw_correct.csv"
crops = crop_extractor(filenames, images_dir, labels_dir, csv_name)

40135_E1_x4300_-10um_0013.png


In [ ]:
def extract_pixel_features(
    image,
    labels,
    stats,
    sigma_fractions=(0.10, 0.20, 0.35),
    truncate=4.0,
):
    """
    Extract multiscale features for pixel classification.

    Features at each sigma:
        - Gaussian-smoothed intensity
        - Sobel edge magnitude
        - Laplacian of Gaussian
        - Hessian eigenvalue 1
        - Hessian eigenvalue 2

    Raw image intensity is also included.

    Parameters
    ----------
    image : np.ndarray, shape (Y, X)
        Grayscale image.

    labels : np.ndarray, shape (Y, X)
        Pixel-wise class labels.

    stats : dict
        Dictionary containing class statistics, e.g.
        stats[class_name]["minor"]["mean"].

    sigma_fractions : tuple
        Fractions of the characteristic object width used for sigma.

    truncate : float
        Radius of Gaussian kernel in units of sigma.
        scipy's normal default is 4 sigma.

    Returns
    -------
    feature_map : np.ndarray
        Shape (Y, X, C).

    data : np.ndarray
        Shape (Y*X, C+1).
        Each row corresponds to one pixel.
        Final column contains its label.

    feature_names : list
        Name of each feature channel.

    sigmas : np.ndarray
        Sigma values used.

    kernel_sizes : np.ndarray
        Approximate kernel size corresponding to each sigma.
    """

    image = np.asarray(image, dtype=np.float32)
    labels = np.asarray(labels)

    if image.ndim != 2:
        raise ValueError("image must be a 2D grayscale image")

    if labels.shape != image.shape:
        raise ValueError("image and labels must have the same shape")

    # ------------------------------------------------------------
    # 1. Determine characteristic object width
    # ------------------------------------------------------------

    minor_means = np.array([
        class_stats["minor"]["mean"]
        for class_stats in stats.values()
    ])

    # Median makes this less influenced by one unusually wide class.
    characteristic_width = np.median(minor_means)

    # ------------------------------------------------------------
    # 2. Choose sigma values
    # ------------------------------------------------------------

    sigmas = characteristic_width * np.asarray(sigma_fractions)

    # Corresponding Gaussian kernel sizes.
    #
    # Radius = truncate * sigma
    # Width = 2 * radius + 1
    #
    kernel_sizes = (
        2 * np.ceil(truncate * sigmas).astype(int) + 1
    )

    # ------------------------------------------------------------
    # 3. Build feature map
    # ------------------------------------------------------------

    features = []
    feature_names = []

    # Raw intensity
    features.append(image)
    feature_names.append("intensity")

    for sigma in sigmas:

        # ---------- Gaussian ----------
        gaussian = gaussian_filter(
            image,
            sigma=sigma,
            truncate=truncate,
        )

        features.append(gaussian)
        feature_names.append(f"gaussian_sigma_{sigma:.2f}")

        # ---------- Sobel ----------
        #
        # Sobel is applied after smoothing so that the edge measurement
        # corresponds to the selected spatial scale.
        sobel_feature = sobel(gaussian)

        features.append(sobel_feature)
        feature_names.append(f"sobel_sigma_{sigma:.2f}")

        # ---------- Laplacian of Gaussian ----------
        log_feature = gaussian_laplace(
            image,
            sigma=sigma,
            truncate=truncate,
        )

        # Scale normalization makes responses at different sigma
        # values more comparable.
        log_feature = (sigma ** 2) * log_feature

        features.append(log_feature)
        feature_names.append(f"LoG_sigma_{sigma:.2f}")

        # ---------- Hessian ----------
        #
        # Second derivatives of the Gaussian-smoothed image.
        Hyy = gaussian_filter(
            image,
            sigma=sigma,
            order=(2, 0),
            truncate=truncate,
        )

        Hxy = gaussian_filter(
            image,
            sigma=sigma,
            order=(1, 1),
            truncate=truncate,
        )

        Hxx = gaussian_filter(
            image,
            sigma=sigma,
            order=(0, 2),
            truncate=truncate,
        )

        # Scale-normalize second derivatives
        Hxx *= sigma ** 2
        Hxy *= sigma ** 2
        Hyy *= sigma ** 2

        # Eigenvalues of a symmetric 2x2 Hessian matrix:
        #
        # [Hxx Hxy]
        # [Hxy Hyy]
        #
        trace = Hxx + Hyy
        difference = Hxx - Hyy

        root = np.sqrt(
            difference ** 2 +
            4 * Hxy ** 2
        )

        eig1 = (trace + root) / 2
        eig2 = (trace - root) / 2

        features.append(eig1)
        feature_names.append(f"hessian_eig1_sigma_{sigma:.2f}")

        features.append(eig2)
        feature_names.append(f"hessian_eig2_sigma_{sigma:.2f}")

    # ------------------------------------------------------------
    # 4. Stack into (Y, X, C)
    # ------------------------------------------------------------

    feature_map = np.stack(features, axis=-1)

    # ------------------------------------------------------------
    # 5. Convert pixels into rows
    # ------------------------------------------------------------

    n_features = feature_map.shape[-1]

    X = feature_map.reshape(-1, n_features)
    y = labels.reshape(-1, 1)

    # Last column = label
    data = np.concatenate((X, y), axis=1)

    return feature_map, data, feature_names, sigmas, kernel_sizes

In [ ]:
def extract_all_crop_features(crops, stats, data_dir):
    """
    Extract pixel features from every crop and combine them into one array.

    Parameters
    ----------
    crops : dict
        Structure:

        crops[data_file]["img_crops"]
        crops[data_file]["labels_crops"]

        Each contains a list of corresponding image/label crops.

    stats : dict
        Object statistics used by extract_pixel_features().

    data_dir : str or Path
        Directory in which to save the feature data.

    Returns
    -------
    all_data : np.ndarray
        All pixel features from all crops concatenated together.
        Final column contains the pixel label.

    row_ranges : dict
        Maps each data filename to its row range in all_data.

    feature_names : list
        Names of the feature columns.
    """

    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    all_crop_data = []
    row_ranges = {}

    current_row = 0
    feature_names = None

    for data_file, crop_data in crops.items():

        img_crops = crop_data["img_crops"]
        labels_crops = crop_data["labels_crops"]

        if len(img_crops) != len(labels_crops):
            raise ValueError(
                f"{data_file}: number of image crops and label crops do not match"
            )

        file_start = current_row

        for img_crop, labels_crop in zip(img_crops, labels_crops):

            (
                feature_map,
                pixel_data,
                names,
                sigmas,
                kernel_sizes,
            ) = extract_pixel_features(
                img_crop,
                labels_crop,
                stats,
            )

            all_crop_data.append(pixel_data)

            current_row += pixel_data.shape[0]

            if feature_names is None:
                feature_names = names

        file_end = current_row

        row_ranges[data_file] = {
            "start": file_start,
            "end": file_end,
        }

    # Combine all pixels from all crops
    all_data = np.concatenate(all_crop_data, axis=0)

    # Save feature + label array
    np.save(
        data_dir / "pixel_features.npy",
        all_data,
    )

    # Save row ranges
    with open(data_dir / "pixel_feature_row_ranges.json", "w") as f:
        json.dump(row_ranges, f, indent=4)

    # Also useful to save the feature column names
    with open(data_dir / "pixel_feature_names.json", "w") as f:
        json.dump(feature_names, f, indent=4)

    return all_data, row_ranges, feature_names

In [ ]:
all_data, row_ranges, feature_names = extract_all_crop_features(
    crops=crops,
    stats=stats,
    data_dir=data_dir,
)

print("Feature array shape:", all_data.shape)
print("Number of features:", len(feature_names))
print("Saved row ranges for:", list(row_ranges.keys()))